In [1]:
import json
import uuid
import random
from datetime import datetime, timedelta
import os 
import sys 
import pandas as pd
import numpy as np
import requests
import zipfile
import xlsxwriter
import  psycopg2
from sqlalchemy import create_engine, inspect
import sqlalchemy as sa
import openpyxl
import argparse 

In [2]:
file_excel = r'unavailable_sessions_.xlsx'


In [12]:
excel_file = pd.ExcelFile(file_excel )
print( dir(excel_file))


['ODFReader', 'OpenpyxlReader', 'PyxlsbReader', 'XlrdReader', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__fspath__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_engines', '_io', '_reader', 'book', 'close', 'engine', 'io', 'parse', 'sheet_names', 'storage_options']


In [15]:
sheets_name = excel_file.sheet_names
print( sheets_name )

['Search by session ID', 'Search by session path', 'Unavailable sessions', 'SMS update history', 'FMS removal history', 'Marker update history', 'Image metadata removal history', 'Thumbnails removal history', 'Erasure coding volumes']


In [18]:
sheets_names_dict = {}
for name in sheets_name:
    name_temp = name.lower()
    name_temp = name_temp.replace(" ","_")
    sheets_names_dict[name_temp] = name
    

In [19]:
print( sheets_names_dict)

{'search_by_session_id': 'Search by session ID', 'search_by_session_path': 'Search by session path', 'unavailable_sessions': 'Unavailable sessions', 'sms_update_history': 'SMS update history', 'fms_removal_history': 'FMS removal history', 'marker_update_history': 'Marker update history', 'image_metadata_removal_history': 'Image metadata removal history', 'thumbnails_removal_history': 'Thumbnails removal history', 'erasure_coding_volumes': 'Erasure coding volumes'}


In [20]:
def create_dataframe_from_excel_sheet(file_name, name_sheet):
    result = pd.read_excel(file_name, sheet_name=name_sheet)
    return result



In [21]:
a1 = create_dataframe_from_excel_sheet(file_excel, 'Erasure coding volumes')

In [29]:
print( a1.columns)
print( a1.shape)
print( a1.info(verbose=True) )


Index(['path', 'vin', 'session_id', 'recording_date', 'volume_type'], dtype='object')
(19098, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19098 entries, 0 to 19097
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   path            19098 non-null  object        
 1   vin             19098 non-null  object        
 2   session_id      19098 non-null  object        
 3   recording_date  19098 non-null  datetime64[ns]
 4   volume_type     19098 non-null  object        
dtypes: datetime64[ns](1), object(4)
memory usage: 746.1+ KB
None


In [31]:
column_data_types = a1[a1.columns].dtypes
print( f'{type(column_data_types)} \n{column_data_types}')

<class 'pandas.core.series.Series'> 
path                      object
vin                       object
session_id                object
recording_date    datetime64[ns]
volume_type               object
dtype: object


In [32]:
data = {
    'int_col': [1, 2, 3, 4, 5],
    'float_col': [1.1, 2.2, 3.3, 4.4, 5.5],
    'str_col': ['apple', 'banana', 'cherry', 'date', 'elderberry'],
    'bool_col': [True, False, True, False, True],
    'datetime_col': pd.to_datetime(['2022-01-01', '2022-02-01', '2022-03-01', '2022-04-01', '2022-05-01']),
    'timedelta_col': [pd.Timedelta(days=1), pd.Timedelta(days=2), pd.Timedelta(days=3), pd.Timedelta(days=4), pd.Timedelta(days=5)],
    'category_col': pd.Categorical(['cat', 'dog', 'dog', 'bird', 'cat']),
    'object_col': [1, 'string', 3.14, True, None],
}

df = pd.DataFrame(data)

# Display the DataFrame and its data types
print(df)
print(df.dtypes)

   int_col  float_col     str_col  bool_col datetime_col timedelta_col  \
0        1        1.1       apple      True   2022-01-01        1 days   
1        2        2.2      banana     False   2022-02-01        2 days   
2        3        3.3      cherry      True   2022-03-01        3 days   
3        4        4.4        date     False   2022-04-01        4 days   
4        5        5.5  elderberry      True   2022-05-01        5 days   

  category_col object_col  
0          cat          1  
1          dog     string  
2          dog       3.14  
3         bird       True  
4          cat       None  
int_col                    int64
float_col                float64
str_col                   object
bool_col                    bool
datetime_col      datetime64[ns]
timedelta_col    timedelta64[ns]
category_col            category
object_col                object
dtype: object


In [33]:
types = {
    'int64': 'INTEGER',
    'float64': 'FLOAT',
    'object': 'TEXT',
    'bool': 'BOOL',
    'datetime64': 'DATE',
    'timedelta64': [pd.Timedelta(days=1), pd.Timedelta(days=2), pd.Timedelta(days=3), pd.Timedelta(days=4), pd.Timedelta(days=5)],
    'category': 'TEXT',
    'object_col': [1, 'string', 3.14, True, None],
}

df = pd.DataFrame(types)

# Display the DataFrame and its data types
print(df)
print(df.dtypes)

     int64 float64 object  bool datetime64 timedelta64 category object_col
0  INTEGER   FLOAT   TEXT  BOOL       DATE      1 days     TEXT          1
1  INTEGER   FLOAT   TEXT  BOOL       DATE      2 days     TEXT     string
2  INTEGER   FLOAT   TEXT  BOOL       DATE      3 days     TEXT       3.14
3  INTEGER   FLOAT   TEXT  BOOL       DATE      4 days     TEXT       True
4  INTEGER   FLOAT   TEXT  BOOL       DATE      5 days     TEXT       None
int64                   object
float64                 object
object                  object
bool                    object
datetime64              object
timedelta64    timedelta64[ns]
category                object
object_col              object
dtype: object
